# 05 · 防控策略优化

本 Notebook 用于：
1. 三场景（常态/散发/聚集）基准对比
2. 7类干预措施效果分析
3. 网格搜索最优防控组合
4. 成本-效益 Pareto 前沿分析
5. 输出论文防控推荐方案表格

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from model.params import ModelParams
from model.solver import solve_seiqr, extract_summary
from model.r0 import compute_R0
from intervention.measures import InterventionBundle, apply_interventions, NO_INTERVENTION, PRESET_MILD, PRESET_MODERATE, PRESET_STRONG
from intervention.scenarios import SCENARIOS, list_scenarios
from intervention.optimizer import grid_search
from intervention.cost_model import compute_cost, compute_economic_cost_yuan
from plots._style import setup_style
from plots.intervention_compare import plot_scenario_comparison, plot_intervention_heatmap, plot_pareto_frontier

setup_style()

FIG_DIR = Path('../output/figures')
OPT_DIR = Path('../output/optimization')
FIG_DIR.mkdir(parents=True, exist_ok=True)
OPT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
p = ModelParams.from_yaml('../config.yaml') if Path('../config.yaml').exists() else ModelParams()
list_scenarios()

## 1. 三场景基准对比

In [ ]:
sim_results = {}
summaries   = {}

for sc_name, scenario in SCENARIOS.items():
    p_sc = scenario.apply_to(p)
    df   = solve_seiqr(p_sc)
    summ = extract_summary(df, p_sc)
    sim_results[scenario.name] = df
    summaries[sc_name] = summ
    R0 = compute_R0(p_sc)
    print(f"[{sc_name}] R₀={R0:.3f} | 峰值={summ['peak_I_rate']:.2%} (第{summ['peak_day']:.0f}天) | AR={summ['total_attack_rate']:.2%}")

fig = plot_scenario_comparison(
    sim_results, metric='I_rate',
    title='三场景流感传播对比（无干预）',
    output_path=FIG_DIR / '05_scenario_baseline.png',
    show=True
)

## 2. 单项干预措施效果对比

In [ ]:
# 以场景二（散发）为基准，逐一测试各项措施
sc_outbreak = SCENARIOS['outbreak']
p_oc = sc_outbreak.apply_to(p)
df_no = solve_seiqr(p_oc)
summ_no = extract_summary(df_no, p_oc)

single_measures = {
    '无干预（基准）':    InterventionBundle(),
    'u1: 口罩（全强度）': InterventionBundle(mask_level=1.0),
    'u2: 通风（全强度）': InterventionBundle(ventilation=1.0),
    'u3: 疫苗（50%覆盖）': InterventionBundle(vaccination=1.0),
    'u4: 隔离强化':      InterventionBundle(isolation_rate=1.0),
    'u5: 线上教学':      InterventionBundle(online_teaching=1.0),
    'u6: 社团限流':      InterventionBundle(activity_limit=1.0),
    'u7: 环境消毒':      InterventionBundle(disinfection=1.0),
}

results_single = {}
for name, bundle in single_measures.items():
    p_iv = apply_interventions(p_oc, bundle)
    df_iv = solve_seiqr(p_iv)
    summ_iv = extract_summary(df_iv, p_iv)
    results_single[name] = {
        '累计发病率':   f"{summ_iv['total_attack_rate']:.2%}",
        '峰值感染率':   f"{summ_iv['peak_I_rate']:.2%}",
        '峰值时间(天)': f"{summ_iv['peak_day']:.0f}",
        '成本评分':     f"{compute_cost(bundle):.3f}",
        'AR降低':       f"{(1 - summ_iv['total_attack_rate']/summ_no['total_attack_rate'])*100:.1f}%",
    }
    results_single[name]['df'] = df_iv

# 打印汇总表
rows = []
for name, res in results_single.items():
    row = {'方案': name}
    row.update({k: v for k, v in res.items() if k != 'df'})
    rows.append(row)

df_table = pd.DataFrame(rows)
print(df_table.to_string(index=False))

# 流行曲线对比
sim_comp = {name: res['df'] for name, res in results_single.items()}
fig = plot_scenario_comparison(
    sim_comp, metric='I_rate',
    title='单项干预措施效果对比（场景二：散发）',
    output_path=FIG_DIR / '05_single_measures.png',
    show=True
)

## 3. 组合方案优化（网格搜索）

In [ ]:
# 场景二（散发）网格搜索
# n_levels=3 共 3^7=2187 种组合，约需 2-3 分钟
print('网格搜索（场景二：散发）...')
df_opt = grid_search(
    p, SCENARIOS['outbreak'],
    n_levels=3,
    cost_limit=0.65,
    output_dir=OPT_DIR,
    top_k=20,
)

if not df_opt.empty:
    print(f'\nTop 10 最优方案:')
    cols_show = ['F_objective', 'AR_reduction_pct', 'PIP_reduction_pct', 'cost_score',
                 'u1_mask', 'u4_isolation', 'u5_online', 'u3_vaccination']
    cols_show = [c for c in cols_show if c in df_opt.columns]
    print(df_opt[cols_show].head(10).round(3).to_string(index=False))

In [ ]:
# 效果热力图
if not df_opt.empty:
    df_opt['scheme_name'] = [f'方案{i+1}' for i in range(len(df_opt))]
    fig = plot_intervention_heatmap(
        df_opt,
        title='防控效果热力图（场景二：散发）',
        output_path=FIG_DIR / '05_intervention_heatmap.png',
        show=True
    )

    # Pareto 前沿
    if 'cost_score' in df_opt.columns and 'AR_reduction_pct' in df_opt.columns:
        fig = plot_pareto_frontier(
            df_opt,
            title='成本-效益 Pareto 前沿（场景二：散发）',
            output_path=FIG_DIR / '05_pareto_frontier.png',
            show=True
        )

## 4. 经济成本估算

In [ ]:
print('预设防控方案经济成本估算（元/学期）:')
presets = {
    '轻度干预':   PRESET_MILD,
    '中度干预':   PRESET_MODERATE,
    '强力干预':   PRESET_STRONG,
}

for name, bundle in presets.items():
    cost_score = compute_cost(bundle)
    econ_cost  = compute_economic_cost_yuan(bundle)
    p_iv = apply_interventions(p, bundle)
    df_iv = solve_seiqr(p_iv)
    summ_iv = extract_summary(df_iv, p_iv)
    baseline_AR = summaries.get('baseline', {}).get('total_attack_rate', 0.15)
    ar_reduce = (1 - summ_iv['total_attack_rate'] / max(baseline_AR, 0.001)) * 100
    
    print(f'\n[{name}]')
    print(f'  成本评分: {cost_score:.3f}')
    print(f'  估算经济成本: {econ_cost/10000:.1f} 万元/学期')
    print(f'  累计发病率: {summ_iv["total_attack_rate"]:.2%}  (降低 {ar_reduce:.1f}%)')
    print(f'  峰值感染率: {summ_iv["peak_I_rate"]:.2%}  峰值时间: 第{summ_iv["peak_day"]:.0f}天')